# Why did the investigator pick that tool? An attribution graph on an open model

This notebook takes one recorded Lattice investigator step, the first tool it chose for a
peeling-chain alert, and asks a small open model the same question as a multiple choice. It then
uses [circuit-tracer](https://github.com/safety-research/circuit-tracer) to build an
**attribution graph** for the answer: which transcoder features, on which prompt tokens, push
the model towards the letter of the recorded tool.

What this is and isn't:

- It explains the **open model's** prediction on a restated prompt. It does not explain the
  production investigator, which runs on a hosted model whose internals cannot be traced.
- It is a showcase of the method (Ameisen et al. 2025, "Circuit Tracing"). It is never part of
  a verdict. The evidence circuit and the ledger remain the system of record.
- Attribution graphs are approximations: the transcoders do not reconstruct the model exactly.
  The panel reports how much of the answer's direct input comes from reconstruction error.

Inputs: `step.json`, written by `python -m lattice.interp.tool_choice`. Outputs: a graph file
under `graphs/`, and `ui/public/interp/tool-choice.json` for the UI's "Inside the model" panel.
See `README.md` for setup and hardware.

## 1. Setup

In [ ]:
# Run from research/interp with the environment described in README.md:
#   circuit-tracer (from its git repository), torch with CUDA, and access to the model weights.
import json
import sys
from pathlib import Path

import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files

HERE = Path.cwd()
REPO = HERE.parents[1]  # the Lattice repository root
sys.path.insert(0, str(REPO / "src"))
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. The recorded step

In [ ]:
step = json.loads((HERE / "step.json").read_text())
print(step["prompt"])
print()
print("recorded tool:", step["recorded_tool"], "-> answer", repr(step["target"]))
OPTION_TOKENS = [f" {o['letter']}" for o in step["options"]]

## 3. The model

`qwen3-0.6b` is the smallest supported model, and its weights are not gated. `gemma-2-2b`
needs about 15 GB of GPU memory with offloading and a Hugging Face login with Gemma's licence
accepted.

In [ ]:
MODELS = {
    "qwen3-0.6b": ("Qwen/Qwen3-0.6B", "mwhanna/qwen3-0.6b-transcoders-lowl0"),
    "gemma-2-2b": ("google/gemma-2-2b", "gemma"),
}
CHOICE = "qwen3-0.6b"
model_name, transcoders = MODELS[CHOICE]

model = ReplacementModel.from_pretrained(model_name, transcoders, dtype=torch.bfloat16)

## 4. What does the model itself answer?

In [ ]:
tokenizer = model.tokenizer
option_ids = [tokenizer.encode(t, add_special_tokens=False) for t in OPTION_TOKENS]
assert all(len(ids) == 1 for ids in option_ids), f"options are not single tokens: {option_ids}"

with torch.no_grad():
    logits = model(step["prompt"])[0, -1].float()
probs = torch.softmax(logits, dim=-1)
for option, ids in zip(step["options"], option_ids):
    mark = "  <- recorded" if option["tool"] == step["recorded_tool"] else ""
    print(f"{option['letter']}) {option['tool']:<22} p = {probs[ids[0]].item():.3f}{mark}")

## 5. Attribution graph

The targets are the option letters, so the graph keeps a logit node for every option, each with
its probability. Lower `max_feature_nodes` or `batch_size` if memory runs out.

In [ ]:
try:
    graph = attribute(prompt=step["prompt"], model=model, attribution_targets=OPTION_TOKENS,
                      batch_size=64, max_feature_nodes=4096, offload="cpu", verbose=True)
except TypeError:
    # Older circuit-tracer releases have no attribution_targets: attribute the top logits.
    graph = attribute(prompt=step["prompt"], model=model, max_n_logits=10,
                      desired_logit_prob=0.95, batch_size=64, max_feature_nodes=4096,
                      offload="cpu", verbose=True)

slug = f"tool-choice-{step['case_id']}-{CHOICE}"
(HERE / "graphs").mkdir(exist_ok=True)
graph_pt = HERE / "graphs" / f"{slug}.pt"
graph.to_pt(graph_pt)
create_graph_files(graph_or_path=graph_pt, slug=slug, output_path=str(HERE / "graphs"),
                   node_threshold=0.8, edge_threshold=0.98)
print("graph file:", HERE / "graphs" / f"{slug}.json")

## 6. Panel summary for the UI

In [ ]:
from lattice.interp.graph_summary import main as summarise

summarise([str(HERE / "graphs" / f"{slug}.json"), "--step", str(HERE / "step.json"),
           "--out", str(REPO / "ui" / "public" / "interp" / "tool-choice.json")])

## 7. Explore the full graph (optional)

circuit-tracer ships an interactive viewer. Open it, pin the logit of the recorded answer, and
follow its strongest inputs back to the prompt tokens.

In [ ]:
from circuit_tracer.frontend.local_server import serve

server = serve(data_dir=str(HERE / "graphs"), port=8046)
print("open http://localhost:8046/index.html")
# server.stop()